<a href="https://colab.research.google.com/github/regmiresearch/ImageProcessingProjects/blob/main/Chapter16/CLIP_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture

import os
if not os.path.exists("MCVP2e-CLIP"):
  !git clone https://github.com/sizhky/MCVP2e-CLIP.git
  %pip install -r MCVP2e-CLIP/requirements.txt

In [2]:
%cd MCVP2e-CLIP

/content/MCVP2e-CLIP


In [4]:
pip install torch_snippets

  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
  Using cached typing-3.7.4.3-py3-none-any.whl
  Using cached jsonlines-4.0.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached imgaug-0.4.0-py2.py3-none-any.whl.metadata (1.8 kB)
  Using cached xmltodict-1.0.4-py3-none-any.whl.metadata (14 kB)
  Using cached fuzzywuzzy-0.18.0-py2.py3-none-any.whl.metadata (4.9 kB)
  Using cached python_levenshtein-0.27.3-py3-none-any.whl.metadata (3.9 kB)
  Using cached pre_commit-4.6.0-py2.py3-none-any.whl.metadata (1.2 kB)
  Using cached asttokens-3.0.1-py3-none-any.whl.metadata (4.9 kB)
  Using cached cfgv-3.5.0-py2.py3-none-any.whl.metadata (8.9 kB)
  Using cached identify-2.6.19-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached nodeenv-1.10.0-py2.py3-none-any.whl.metadata (24 kB)
  Using cached virtualenv-21.5.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached levenshtein-0.27.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (3.7 kB)
  Using cached rapidfuzz-

In [ ]:
import itertools
from torch_snippets import *
from clip.core import download_flickr8k_from_kaggle
from clip.config import ClipConfig
from clip.dataset import CLIPDataset
from clip.models import CLIP

In [ ]:
%%writefile kaggle.json
{"username":"XXX","key":"XXX"}

Before running the next line of code,
ensure you are agreeing to kaggle’s terms in the webpage -
https://www.kaggle.com/datasets/adityajn105/flickr8k once.
Else the download wouldn’t happen

In [ ]:
kaggle_json_path = P("kaggle.json")
data_download_path = P("/content/flickr-8k-kaggle/")
download_flickr8k_from_kaggle(kaggle_json_path, data_download_path)

df = pd.read_csv(data_download_path / "captions.txt")
df["id"] = [id_ for id_ in range(len(df) // 5) for _ in range(5)]
df.to_csv(data_download_path / "captions.csv")

In [ ]:
df.head()

In [ ]:
config = ClipConfig()
config.image_path = data_download_path / "Images"
config.captions_csv_path = data_download_path / "captions.csv"
# Make any other experiment changes, if you want to, below
config.debug = False  # Switch to True, in case you want to reduce the dataset size
config.epochs = 1
config.save_eval_and_logging_steps = 50

In [ ]:
config

In [ ]:
trn_ds, val_ds = CLIPDataset.train_test_split(config)

model = CLIP(config).to(config.device)

params = [
    {"params": model.image_encoder.parameters(), "lr": config.image_encoder_lr},
    {"params": model.text_encoder.parameters(), "lr": config.text_encoder_lr},
    {
        "params": itertools.chain(
            model.image_projection.parameters(), model.text_projection.parameters()
        ),
        "lr": config.head_lr,
        "weight_decay": config.weight_decay,
    },
]

optimizer = torch.optim.AdamW(params, weight_decay=0.0)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=config.patience, factor=config.factor
)

In [ ]:
from transformers import Trainer, TrainingArguments

# Define TrainingArguments
training_args = TrainingArguments(
    output_dir="./results",  # Output directory where checkpoints and logs will be saved.
    num_train_epochs=config.epochs,  # Total number of training epochs.
    per_device_train_batch_size=config.batch_size,  # Batch size per GPU.
    per_device_eval_batch_size=config.batch_size,  # Batch size for evaluation per GPU.
    evaluation_strategy="steps",  # Evaluation strategy (steps, epoch).
    logging_strategy="steps",  # Logging strategy (steps, epoch).
    save_strategy="steps",  # Save strategy (steps, epoch).
    save_total_limit=2,  # Limit the total amount of checkpoints.
    learning_rate=5e-5,  # Learning rate.
    logging_steps=config.save_eval_and_logging_steps,
    save_steps=config.save_eval_and_logging_steps,  # Save checkpoints every N steps.
    eval_steps=config.save_eval_and_logging_steps,  # Evaluate every N steps.
    logging_dir="./logs",  # Directory for storing logs.
    metric_for_best_model="loss",
    label_names=["image", "input_ids"],
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=trn_ds,
    eval_dataset=val_ds,
    optimizers=(optimizer, lr_scheduler),  # Pass your custom optimizer here.
)

# Train the model
trainer.train()

In [ ]:
from clip.infer import get_image_embeddings, find_matches

# in case you want to load a pretrained model...
# model = CLIP.from_pretrained("results/checkpoint-550/", config)
# _ = model.to(config.device)

val_dl = DataLoader(
    val_ds,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    shuffle=False,
)

image_embeddings = get_image_embeddings(val_dl, model)

In [ ]:
query="dog playing on grass"
find_matches(model, image_embeddings, query=query, image_filenames=val_dl.dataset.image_filenames)

In [ ]:
1